# 🎭 The canonical example — Ada, Bell, and ticket #7

One concrete sale — Ada buys **50 Mbps** from Bell for **10 TOK**, minted as ticket
**#7** — is carried by the *entire* repo: the story quotes it, the formal docs quote it,
every test imports it, and every notebook (including this one) runs on it. This chapter
dissects **every value** in that example, and uses each one to teach the concept
underneath it.

**What you'll be able to do after this notebook**

- explain what an Ethereum address, a unix timestamp, a hash, and an ABI blob actually *are* — byte by byte
- do integer-money arithmetic (10 TOK = `10_000_000_000_000_000_000` base units) and say why floats never touch money
- decode the on-chain `params` blob **by hand** — no library — and recover `50_000_000` and `1`
- diff the signed `Offer` against the minted `EntitlementView` and explain every field that appears or disappears
- state why ONE shared example is a *hard convention*, and prove with `grep` that the repo obeys it

**You need:** notebooks 00–03 (the story, Python constructs, pydantic, Protocols).
**Infra:** none — this chapter runs fully offline.

**Estimated time:** ~60 minutes.

> **How to run:** ▶▶ Run All, or step through with Shift+Enter. Each markdown cell tells
> you what to look for in the code cell below it.

## 1. One example, everywhere — why this is a hard rule

A project tells the same story in many places: a narrative doc, formal schemas, tests,
notebooks. If each place invents its own numbers ("Alice buys 100 Mbps for 5 TOK…"),
then the day one doc says the window ends at 16:00 and a test says 15:00, *which is
right?* A **divergent value is a whole bug class** — it can hide a real disagreement
about behavior behind what looks like sloppy copy-editing.

So the repo makes it a rule. From `CLAUDE.md` (the repo's binding conventions):

> Canonical example values (Ada/Bell/ticket #7/10 TOK/50 Mbps) come from
> `a2a_interfaces.fixtures` — story, docs, and tests share them; change them in one
> place or not at all. Prose/schemas may *quote* these values but never introduce a
> divergent one.

The single source is one module: `interfaces/src/a2a_interfaces/fixtures.py`. Import it
and read its own docstring — the rule is stated right at the top.

In [ ]:
from a2a_interfaces import fixtures as fx   # the single source of truth for the example

print(fx.__doc__)

**Prove it live.** Don't take the rule on faith — grep the repo from inside the
notebook. We locate the repo root (course notebooks run two folders deep), then count
how often the docs quote Ada's address prefix `0xf39F`, the phrase `ticket #7`, and
`50 Mbps`. Expect *dozens* of hits: the docs quote the canon constantly.

In [ ]:
import subprocess
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

def hits(pattern: str, *paths: str) -> int:
    """How many times `pattern` occurs under ROOT (grep -ro lists every single match)."""
    out = subprocess.run(["grep", "-ro", pattern, *paths],
                         cwd=ROOT, capture_output=True, text=True).stdout
    return len(out.splitlines())

counts = {p: hits(p, "docs") for p in ["0xf39F", "ticket #7", "50 Mbps"]}
for pattern, n in counts.items():
    print(f"docs/ quotes {pattern!r:12} → {n:3d} times")
assert counts["0xf39F"] >= 5 and counts["ticket #7"] >= 10 and counts["50 Mbps"] >= 10

Docs may *quote*; tests are held one step stricter — they **import**. If a doc drifts,
a human reviewer can catch the prose; a test that hard-coded its own address would only
be caught by luck. Grep every package's `tests/` folder: Ada's address appears verbatim
**zero** times, while the word `fixtures` appears in ~two dozen test files.

In [ ]:
test_dirs = sorted(str(p.relative_to(ROOT)) for p in ROOT.glob("*/tests"))
print("test dirs:", test_dirs, "\n")

verbatim = subprocess.run(["grep", "-ro", "--include=*.py", "0xf39F", *test_dirs],
                          cwd=ROOT, capture_output=True, text=True).stdout.splitlines()
importing = subprocess.run(["grep", "-rl", "--include=*.py", "fixtures", *test_dirs],
                           cwd=ROOT, capture_output=True, text=True).stdout.splitlines()
print(f"'0xf39F' hard-coded in tests → {len(verbatim):2d} times   ← never copy-pasted")
print(f"test files using fixtures   → {len(importing):2d}, e.g. {importing[:3]}")
assert len(verbatim) == 0 and len(importing) >= 10

**✏️ Your turn 1.1 — grep the canon's price**

The cells above counted the address, the ticket, and the bandwidth — but the **price**
is canon too. Point the same `hits()` helper at `"10 TOK"` and rerun. Success: dozens
of hits, and the un-commented assert passes (it wants at least 10).

In [ ]:
pattern = "0xf39F"                    # TODO: change the pattern to "10 TOK"
n = hits(pattern, "docs")
print(f"docs/ quotes {pattern!r} → {n} times")
# TODO: un-comment the self-check once you've switched the pattern
# assert pattern == "10 TOK" and n >= 10

<details><summary>✅ Solution 1.1 — peek only after trying</summary>

```python
pattern = "10 TOK"
n = hits(pattern, "docs")
print(f"docs/ quotes {pattern!r} → {n} times")
assert pattern == "10 TOK" and n >= 10
```

Every canonical value — not just the three sampled above — is quoted all over the
docs, and none may diverge from `fixtures.py`.

</details>

**Break it on purpose.** The package's front door (`from a2a_interfaces import …`)
exports only wire shapes and ports — the "published language" you met in 02 and 03.
Example *data* deliberately lives one door over, in `a2a_interfaces.fixtures`, so demo
numbers can never blur into the wire-shape surface. Watch the wrong door fail:

In [ ]:
try:
    from a2a_interfaces import CANONICAL_OFFER          # wrong door
except ImportError as e:
    print("✗ ImportError:", e)

from a2a_interfaces.fixtures import CANONICAL_OFFER     # right door
print("✓ fixtures live in their own namespace: a2a_interfaces.fixtures")

## 2. The cast: `ADA` and `BELL` — what an address *is*

**Recap so far:** one module holds the canon; everything else quotes or imports it.
Now tour the values, starting with the two players.

An **address** is an account's public name on a blockchain: exactly **20 bytes**,
written as 40 hexadecimal characters behind a `0x` prefix. There is no username and no
registry — the address *is* the identity, and it is **derived from a private key** (a
secret random number; the full derivation, key → address, opens chapter 06). Whoever
holds the key controls the address; everyone else can only send *to* it.

Ada's and Bell's addresses aren't invented: they are the first two accounts of **Anvil**,
the disposable development chain you'll meet in 06 — chosen so the whole course can
predict them in advance.

In [ ]:
print("ADA  (consumer, buys  #7):", fx.ADA)
print("BELL (provider, sells #7):", fx.BELL)

raw = bytes.fromhex(fx.ADA[2:])        # strip '0x', parse each pair of hex chars as one byte
print(f"→ {len(fx.ADA[2:])} hex chars = {len(raw)} raw bytes")
assert len(raw) == 20 and fx.ADA.startswith("0x")
raw                                     # bytes repr: hex pairs, some shown as ASCII

**✏️ Your turn 2.1 — parse Bell's address into raw bytes**

We just split Ada's address into 20 raw bytes. Do the same for `fx.BELL`: strip the
`0x`, parse with `bytes.fromhex`. Success: 20 bytes that are *not* equal to Ada's —
a different identity, the exact same shape.

In [ ]:
bell_raw = raw                        # TODO: build this from fx.BELL, the way `raw` was built from fx.ADA
print(f"BELL → {len(bell_raw)} raw bytes")
# TODO: un-comment once bell_raw really comes from BELL
# assert bell_raw == bytes.fromhex(fx.BELL[2:]) and len(bell_raw) == 20 and bell_raw != raw

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
bell_raw = bytes.fromhex(fx.BELL[2:])
print(f"BELL → {len(bell_raw)} raw bytes")
assert bell_raw == bytes.fromhex(fx.BELL[2:]) and len(bell_raw) == 20 and bell_raw != raw
```

Every address — Ada's, Bell's, even a contract's (§12) — is the same 20-byte shape;
only the bytes differ.

</details>

Look closer at the spelling: `0xf39Fd6e5…` — why the odd MiXeD cAsE? It's a
**checksum** (EIP-55): the pattern of upper/lowercase letters encodes a fingerprint of
the address itself, so a single mistyped character makes the casing invalid and tools
reject it. Computing that fingerprint needs the keccak hash (hashes: §9; keccak lives in
`chainmcp`), so this package validates only the *pattern* — `0x` + exactly 40 hex chars.

In [ ]:
upper = sum(c.isupper() for c in fx.ADA[2:])
lower = sum(c.islower() for c in fx.ADA[2:])
digits = sum(c.isdigit() for c in fx.ADA[2:])
print(f"ADA's 40 chars: {upper} uppercase + {lower} lowercase + {digits} digits")
print("→ the casing is an embedded checksum (EIP-55), not decoration")
assert upper > 0 and lower > 0 and upper + lower + digits == 40

**Break it on purpose.** The `Address` type (a regex-constrained string — 02's
`StringConstraints`) guards every border: a truncated address can never enter the
system disguised as a real one.

In [ ]:
from pydantic import ValidationError
from a2a_interfaces.models import Offer

try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "provider": "0x123"})
except ValidationError as e:
    err = e.errors()[0]
    print(f"✗ rejected: {err['type']}")
    print("→ 0x + exactly 40 hex chars, or the value never crosses a package boundary")

## 3. `ZERO_ADDRESS` — the address of nobody

Twenty bytes of zero is a perfectly *shaped* address that nobody controls (no known key
derives to it). Chains use it as a **sentinel** — an in-band value meaning "no one".
Here it fills the offer's `consumer` field: `consumer = ZERO_ADDRESS` means the offer is
**open** — whoever pays first gets the ticket. Bell signs a catalog page, not a personal
contract with Ada.

In [ ]:
print("ZERO_ADDRESS  :", fx.ZERO_ADDRESS)
print("offer.consumer:", fx.CANONICAL_OFFER.consumer)
assert fx.ZERO_ADDRESS == "0x" + "0" * 40
assert fx.CANONICAL_OFFER.consumer == fx.ZERO_ADDRESS
print("✓ the canonical offer is OPEN — whoever fulfills first (Ada did) wins the ticket")

## 4. `WINDOW` — unix time from zero

Computers don't store "September 15th, two in the afternoon". They store **unix time**:
the count of seconds since 1970-01-01 00:00 UTC (the *epoch*). One plain integer — no
time zones, no calendar math, and "is it valid yet?" becomes a `<` comparison. The
canonical service window is two such integers. Render them back into the story's clock:

In [ ]:
from datetime import datetime, timezone

print("WINDOW.start:", fx.WINDOW.start)
print("WINDOW.end  :", fx.WINDOW.end)

start = datetime.fromtimestamp(fx.WINDOW.start, tz=timezone.utc)
end = datetime.fromtimestamp(fx.WINDOW.end, tz=timezone.utc)
print(f"→ {start:%Y-%m-%d %H:%M} UTC → {end:%H:%M} UTC   ← the story's '14:00 → 16:00'")
print(f"→ duration: {(fx.WINDOW.end - fx.WINDOW.start) / 3600:.0f} hours")
assert fx.WINDOW.end - fx.WINDOW.start == 7200

**✏️ Your turn 4.1 — predict the window's midpoint**

One hour into the two-hour window is `fx.WINDOW.start + 3600`. **Before running**,
write the clock time (HH:MM UTC) you expect into the prediction comment — then run and
compare. Success: your predicted digits match the printed ones, and the un-commented
assert agrees.

In [ ]:
# my prediction (HH:MM UTC): ...       ← TODO: write it BEFORE you run
midpoint = fx.WINDOW.start + 3600
shown = datetime.fromtimestamp(midpoint, tz=timezone.utc).strftime("%H:%M")
print(f"midpoint {midpoint} → {shown} UTC")
# TODO: un-comment to pin the answer once your prediction agrees
# assert shown == "15:00"

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
# my prediction (HH:MM UTC): 15:00
midpoint = fx.WINDOW.start + 3600
shown = datetime.fromtimestamp(midpoint, tz=timezone.utc).strftime("%H:%M")
print(f"midpoint {midpoint} → {shown} UTC")
assert shown == "15:00"
```

The window starts at 14:00 UTC, and unix time turns "one hour later" into plain
integer addition — `+3600`, no calendar math.

</details>

Why **absolute** times rather than "2 hours from activation"? A duration needs a start
*event*, and events can be argued about ("it started when *I* clicked" vs "when *you*
processed it"). Two absolute timestamps make validity a fact anyone can check against a
shared clock — and the shared clock is **chain time** (`block.timestamp`, ADR-004),
never somebody's laptop clock. Notebook 08's predicate re-checks it before every action.

**Break it on purpose.** Fixtures are frozen (02's `ConfigDict(frozen=True)`). A shared
example that one test could quietly mutate would poison every other test that imports it.

In [ ]:
try:
    fx.WINDOW.start = 0                       # try to bend time
except ValidationError as e:
    print("✗ mutation rejected:", e.errors()[0]["type"])
print("✓ shared fixtures are read-only — variants use model_copy(update=...), met in 02")

## 5. `CAPACITY_50_MBPS` — bits vs bytes

Storage is measured in **bytes** (Ada's 45 GB dataset); links are measured in **bits
per second** (1 byte = 8 bits — networks count what crosses the wire each tick, and the
smallest thing that crosses is a bit). "50 Mbps" = 50 million bits per second, stored as
the plain integer `50_000_000` — the underscores are Python digit separators, pure
readability, zero effect on the value.

The story's Prologue claims 50 Mbps moves 45 GB in exactly the 2-hour window. Check the
arithmetic live instead of believing it:

In [ ]:
print("CAPACITY_50_MBPS:", fx.CAPACITY_50_MBPS)
assert 50_000_000 == 50000000            # underscores change nothing

dataset_bytes = 45 * 10**9                       # 45 GB
window_s = fx.WINDOW.end - fx.WINDOW.start       # 7200 s
bits_moved = fx.CAPACITY_50_MBPS * window_s
print(f"{fx.CAPACITY_50_MBPS:,} bit/s × {window_s:,} s = {bits_moved:,} bits")
print(f"= {bits_moved // 8:,} bytes = {bits_moved // 8 / 10**9:.0f} GB   ← exactly Ada's dataset")
assert bits_moved // 8 == dataset_bytes

**✏️ Your turn 5.1 — halve the pipe**

Change the capacity to `25_000_000` (25 Mbps) and rerun. The Prologue's arithmetic
breaks: how many GB now move in the same 2-hour window? Success: `22.5` GB — Ada's
45 GB dataset no longer fits.

In [ ]:
capacity_bps = fx.CAPACITY_50_MBPS    # TODO: change to 25_000_000 — half the pipe
gb_moved = capacity_bps * window_s // 8 / 10**9
print(f"{capacity_bps:,} bit/s × {window_s:,} s → {gb_moved} GB moved")
# TODO: un-comment after halving — half the pipe moves half the data
# assert gb_moved == 22.5

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
capacity_bps = 25_000_000
gb_moved = capacity_bps * window_s // 8 / 10**9
print(f"{capacity_bps:,} bit/s × {window_s:,} s → {gb_moved} GB moved")
assert gb_moved == 22.5
```

Capacity and window multiply: 45 GB at 50 Mbps needs *exactly* the full 2 hours, so
any thinner pipe fails Ada's need — that is why she shops for 50, not "whatever".

</details>

## 6. `PRICE_10_TOK` — money is an integer

Before looking at the price, watch why floating-point numbers can never carry money —
binary floats cannot represent most decimal fractions exactly:

In [ ]:
print("0.1 + 0.2 =", 0.1 + 0.2, "  ← not 0.3; binary floats approximate decimals")
assert 0.1 + 0.2 != 0.3

Chains therefore store token amounts as **integers of the smallest unit**. TOK — this
project's toy currency, an ERC-20 **token** (a fungible balance a contract keeps; 06
builds the concept properly) — uses 18 decimals: 1 TOK = 10¹⁸ base units. "10 TOK" is
the integer 10·10¹⁸. And in JSON it travels as a **decimal string**, because many JSON
parsers read numbers as floats and would mangle a 20-digit integer. That is exactly the
`DecimalString` type you met in 02.

In [ ]:
print("PRICE_10_TOK:", fx.PRICE_10_TOK, f"  ({type(fx.PRICE_10_TOK).__name__} — a string on the wire)")
price = int(fx.PRICE_10_TOK)
print(f"→ as an integer: {price:,}")
print(f"→ in whole TOK : {price // 10**18} TOK")
assert price == 10 * 10**18

**Break it on purpose.** A decimal point never crosses the border — `DecimalString`'s
pattern is digits only:

In [ ]:
try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "price": "10.5"})
except ValidationError as e:
    print("✗ '10.5' rejected:", e.errors()[0]["type"])
    print("→ integer base units only; '10.5 TOK' must be written '10500000000000000000'")

**✏️ Your turn 6.1 — spell 2.5 TOK legally**

`"10.5"` was just rejected. Write **2.5 TOK** the way the wire wants it: an integer of
base units, digits only, as a string — and use *integer* arithmetic (shift the decimal
into the exponent; the float `2.5` must never appear). Success: both un-commented
checks pass, including the `Offer` border accepting your string.

In [ ]:
two_and_a_half_tok = fx.PRICE_10_TOK  # TODO: replace — 2.5 TOK = 25 × 10**17 base units, as a str
print("on the wire:", two_and_a_half_tok)
# TODO: un-comment both checks when ready
# assert int(two_and_a_half_tok) == 25 * 10**17
# assert Offer(**{**fx.CANONICAL_OFFER.model_dump(), "price": two_and_a_half_tok}).price == two_and_a_half_tok

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
two_and_a_half_tok = str(25 * 10**17)
print("on the wire:", two_and_a_half_tok)
assert int(two_and_a_half_tok) == 25 * 10**17
assert Offer(**{**fx.CANONICAL_OFFER.model_dump(), "price": two_and_a_half_tok}).price == two_and_a_half_tok
```

Shifting the decimal point into the exponent (2.5 × 10¹⁸ = 25 × 10¹⁷) keeps money in
exact integers end to end — the lossy float never exists.

</details>

## 7. `QOS_CLASS` — one small integer for "how good"

Real networks tag traffic with a quality-of-service class — which queue your packets
ride in when a link is busy. The canonical sale asks for class `1`. It's a `Uint8`
(0–255, mirroring the chain's `uint8` width — 02's bounded integers), and it travels
*next to* the capacity inside the params blob you'll dissect in §16. How faithfully the
lab routers *enforce* QoS is 09's honesty section (ADR-006).

In [ ]:
print("QOS_CLASS:", fx.QOS_CLASS)
assert 0 <= fx.QOS_CLASS <= 255        # uint8 range — chain integer widths, met in 02

## 8. `SALT` — the serial number on the coupon

Bell's signed offer is like a signed discount coupon. What stops Ada — or anyone who
intercepted it — from redeeming the *same* coupon five times? (Re-submitting a valid
signed message to get its effect again is called a **replay attack**.) Every offer
carries a **salt**: a unique serial number. The contract keeps a ledger of punched
serials and refuses repeats — an offer is **single-use**. In 05 you'll watch the fake
chain punch this exact salt; in 07 the *real* contract refuses the replay.

The value is one full 32-byte chain word — and its number is a joke: `0x5A17` is
hexspeak for "SALT" (5≈S, A, 1≈L, 7≈T).

In [ ]:
print("SALT:", fx.SALT)
print(f"→ as an integer: {int(fx.SALT, 16):,}   (= 0x5A17, hexspeak for 'SALT')")
assert int(fx.SALT, 16) == 0x5A17 == 23063
assert len(fx.SALT) == 2 + 64          # '0x' + 64 hex chars = 32 bytes

**✏️ Your turn 8.1 — shrink the salt and name the check that fires**

Replace the salt candidate with the bare hexspeak `"0x5A17"` — 2 bytes instead of a
full 32-byte word — and rerun. Success: a `ValidationError` whose `type` names the
pattern check — the same regex guard that caught the truncated address in §2.

In [ ]:
salt_candidate = fx.SALT              # TODO: change to "0x5A17" — 2 bytes, not 32
try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "salt": salt_candidate})
    print("✓ accepted — still the full 32-byte canonical salt")
except ValidationError as e:
    print("✗ rejected:", e.errors()[0]["type"], "  ← the check that fired")
# TODO: un-comment once you've seen the rejection
# assert salt_candidate == "0x5A17"

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
salt_candidate = "0x5A17"
try:
    Offer(**{**fx.CANONICAL_OFFER.model_dump(), "salt": salt_candidate})
    print("✓ accepted — still the full 32-byte canonical salt")
except ValidationError as e:
    print("✗ rejected:", e.errors()[0]["type"], "  ← the check that fired")
assert salt_candidate == "0x5A17"
```

`string_pattern_mismatch` — the 32-byte-hex regex demands `0x` + exactly 64 hex
chars, so a serial number missing 30 bytes never crosses the border.

</details>

## 9. `TERMS_HASH` — hashes from zero

A **hash function** turns *any* input into a fixed-size fingerprint with three
properties: the same input always gives the same output (**deterministic**); a tiny
input change scrambles the entire output (the **avalanche effect**); and it cannot be
run backwards. That makes a hash a perfect *commitment*: publish the fingerprint today,
reveal the document later, and anyone can verify they match — but nobody can craft a
different document with the same fingerprint. Watch all of it with Python's built-in
`hashlib`:

In [ ]:
import hashlib

a = hashlib.sha256(b"50 Mbps, 14:00-16:00, 10 TOK").hexdigest()
b = hashlib.sha256(b"50 Mbps, 14:00-16:00, 10 TOK").hexdigest()
c = hashlib.sha256(b"50 Mbps, 14:00-16:00, 11 TOK").hexdigest()   # ONE character differs

print("same input  :", a[:32], "…")
print("same again  :", b[:32], "…   ← deterministic")
print("1 char later:", c[:32], "…   ← avalanche")
diff = sum(x != y for x, y in zip(a, c))
print(f"→ {diff}/64 hex chars differ after a one-character edit")
assert a == b and a != c and diff > 40

In the repo, the offer's human-readable SLA (`TERMS_DOC` — latency and loss promises)
is hashed, and only the 32-byte fingerprint goes on-chain as `termsHash`: the chain
stores the *commitment*, not the prose. Ethereum's hash function is **keccak-256** (a
close cousin of sha-256) — and per this repo's layering, keccak lives only in
`chainmcp`, because `interfaces` does no hashing at all. So the fixture's `TERMS_HASH`
is a syntactically valid **placeholder** — `22` repeated 32 times. Notebook 07 computes
the real one.

In [ ]:
print("TERMS_DOC :", fx.TERMS_DOC)
print("TERMS_HASH:", fx.TERMS_HASH)
assert fx.TERMS_HASH == "0x" + "22" * 32       # a placeholder, not a real keccak digest
print("✓ placeholder ('22' × 32) — the real keccak256(terms_doc) arrives in notebook 07")

## 10. `RESOURCE_ID` — an opaque name (whose name is it?)

Every entitlement points at *some real thing* — here, the hostA→hostB path. But the
chain stores only an **opaque 32-byte value**. Whose name is it? ADR-005 answers with a
three-way split:

- the **provider** (Bell) *names* it — an id from Bell's own private catalog;
- the **controller** *resolves* it — a private map (`resource_map.yaml`) from id to concrete device and interfaces;
- the **consumer** (Ada) *never interprets* it — to her it's a meaningless token she buys and presents back.

The split keeps topology secret (Ada learns nothing about Bell's network from the id)
and keeps the contract service-agnostic (32 bytes can name a path, a router, or
anything sold next year). The fixture builds it with an f-string *format spec* (you met
f-strings in 01): `f"{TICKET_ID:064x}"` means "render as lowercase hex, zero-padded to
64 characters".

In [ ]:
print("RESOURCE_ID:", fx.RESOURCE_ID)
print("→ built as '0x' + f\"{TICKET_ID:064x}\": the int", fx.TICKET_ID, "zero-padded to 64 hex chars")
assert fx.RESOURCE_ID == "0x" + f"{fx.TICKET_ID:064x}"
assert len(bytes.fromhex(fx.RESOURCE_ID[2:])) == 32
print("✓ 32 opaque bytes — what they resolve TO (a device + two interfaces) appears in §17")

**✏️ Your turn 10.1 — name Bell's second catalog entry**

Build the opaque 32-byte name for Bell's catalog entry **8** (what it resolves to
appears in §17) using the same f-string format spec the fixture uses. Success: 63
zeros ending in `8`, and 32 raw bytes.

In [ ]:
entry_8 = fx.RESOURCE_ID              # TODO: build it yourself: "0x" + f"{8:064x}"
print("entry 8:", entry_8)
# TODO: un-comment when the name is really entry 8's
# assert entry_8 == "0x" + "0" * 63 + "8" and len(bytes.fromhex(entry_8[2:])) == 32

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
entry_8 = "0x" + f"{8:064x}"
print("entry 8:", entry_8)
assert entry_8 == "0x" + "0" * 63 + "8" and len(bytes.fromhex(entry_8[2:])) == 32
```

This is byte-for-byte how `fixtures.py` builds `TELEMETRY_RESOURCE_ID` — the
`:064x` spec does all the padding, so any catalog integer becomes a valid opaque name.

</details>

## 11. `TICKET_ID = 7` — why seven?

When the sale settles, the contract mints an ERC-721 ticket (an **NFT** — a token where
every unit is distinct and individually owned, unlike TOK's interchangeable balance;
06 builds both concepts) and hands it the next id in sequence. Ids count **from 1**, so
ticket #7 means *six sales happened before Ada's*. The canon deliberately lives
mid-sequence: real systems are always mid-history, and code that only works for id 1
is hiding an off-by-one bug. (That Bell's catalog id in §10 is also the number 7 is a
fixture convenience — easy on human eyes in hex dumps, meaningless to the machine.)

In [ ]:
print("TICKET_ID:", fx.TICKET_ID, "  ← the 7th ticket ever minted; ids count from 1")
assert fx.RESOURCE_ID.endswith("7")
assert fx.CANONICAL_ENTITLEMENT_VIEW.id == fx.TICKET_ID   # the minted view carries the same id

**✏️ Your turn 11.1 — count the history around ticket #7**

Before running, fill in the two predictions in the comment: how many tickets existed
*before* Ada's, and what id the *next* mint after hers gets. Success: the un-commented
assert agrees with both.

In [ ]:
# my predictions — tickets before Ada's: ...   id of the next mint: ...   ← TODO: fill in BEFORE running
before, next_id = fx.TICKET_ID - 1, fx.TICKET_ID + 1
print(f"minted before Ada's: {before}   next id after hers: {next_id}")
# TODO: un-comment to pin your predictions
# assert (before, next_id) == (6, 8)

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
# my predictions — tickets before Ada's: 6   id of the next mint: 8
before, next_id = fx.TICKET_ID - 1, fx.TICKET_ID + 1
print(f"minted before Ada's: {before}   next id after hers: {next_id}")
assert (before, next_id) == (6, 8)
```

Ids count from 1, so #7 is the seventh ticket ever — six sales before Ada's, and #8
is next (§17 sells it).

</details>

## 12. `MOCK_TOK` — an address known *before* deployment

`MOCK_TOK` is the address of the payment-token contract (a **smart contract**: code
that lives at an address and holds state — chapter 06, from zero). How can a fixture
hard-code where a contract *will* live? Because contract addresses are computed
**deterministically** from just two inputs: the deployer's address and how many
transactions that deployer has sent so far. On a fresh Anvil chain the same deployer
runs the same deploy script, so MockTOK always lands at the same address — deploy #0.
Notebook 06 verifies this live on a real chain; notebook 11's world-swap depends on it.

In [ ]:
print("MOCK_TOK:", fx.MOCK_TOK)
assert fx.MOCK_TOK == fx.CANONICAL_OFFER.payment_token   # the offer prices itself in this token
assert len(bytes.fromhex(fx.MOCK_TOK[2:])) == 20         # contracts have addresses like any account

## 13. `CANONICAL_OFFER` — twelve questions, twelve answers

You now know every ingredient. Time for the assembled views — first the *question* the
offer answers: `BANDWIDTH_NEED`, what Ada asks Bell. Note `src`/`dst` are names from
**Bell's catalog** (`hostA`, `hostB`) — Ada never sees real topology (§10's split,
already at work).

In [ ]:
print("Ada's need (kind =", fx.BANDWIDTH_NEED.kind + "):")
fx.BANDWIDTH_NEED.model_dump()

Bell's answer is the `Offer`: **twelve fields** that mirror the on-chain EIP-712 struct
field-for-field — what gets signed must equal what the contract verifies (07 proves it
byte-for-byte). Each field answers one question, for one audience:

| # | field | answers… | for whom |
|---|---|---|---|
| 1 | `provider` | who signed this / whom to pay | contract |
| 2 | `consumer` | who may redeem (`0x0…0` = anyone, §3) | contract |
| 3 | `service_type` | which kind of service (0 = bandwidth) | contract stores it; controller picks the translator |
| 4 | `resource_id` | which real-world thing (opaque, §10) | controller resolves it |
| 5 | `params` | how much / what class (ABI blob, §16) | controller decodes it |
| 6 | `start_time` | valid from when (§4) | controller's predicate |
| 7 | `end_time` | valid until when (§4) | controller's predicate |
| 8 | `payment_token` | pay in which currency (§12) | contract |
| 9 | `price` | pay how much, in base units (§6) | contract |
| 10 | `valid_until` | how long the *quote itself* stands | contract |
| 11 | `salt` | the coupon's serial (§8) | contract |
| 12 | `terms_hash` | fingerprint of the SLA (§9) | contract stores it; anyone verifies |

In [ ]:
offer = fx.CANONICAL_OFFER
dump = offer.model_dump()
assert len(dump) == 12
for i, (k, v) in enumerate(dump.items(), 1):
    print(f"{i:2d}. {k:14} = {v}")

Two things to notice. First, **`valid_until` ≠ `end_time`**: the *quote* expires at
14:20 (take it or lose it — Bell won't hold capacity open forever), while the *service*
runs to 16:00. Second, **one source of truth in practice**: the offer's window fields
are read *off the `WINDOW` object* in `fixtures.py`, never typed in again.

In [ ]:
q = datetime.fromtimestamp(offer.valid_until, tz=timezone.utc)
e = datetime.fromtimestamp(offer.end_time, tz=timezone.utc)
print(f"valid_until → {q:%H:%M} UTC   ← the QUOTE's shelf life (Bell signed it ~13:31)")
print(f"end_time    → {e:%H:%M} UTC   ← the SERVICE window's end")
assert offer.valid_until < offer.end_time
assert offer.start_time == fx.WINDOW.start and offer.end_time == fx.WINDOW.end
print("✓ window fields come FROM the WINDOW fixture — derived, never re-typed")

On the wire — and under Bell's signature — the offer is spelled **camelCase**, mirroring
the Solidity struct exactly (the alias trick from 02). Same data, two spellings; the
arrows mark every field whose name changes:

In [ ]:
snake = list(offer.model_dump())
camel = list(offer.model_dump(by_alias=True))
for s, c in zip(snake, camel):
    mark = "→" if s != c else " "
    print(f"  {s:14} {mark} {c}")
assert "serviceType" in camel and "service_type" in snake

**✏️ Your turn 13.1 — predict how many names change spelling**

Scroll back over the arrows above — or reason it out: only names containing `_` can
change. Write your prediction, then count with code. Success: your number matches
`changed`.

In [ ]:
# my prediction: ... of the 12 names change in camelCase   ← TODO: write it BEFORE running
changed = sum(s != c for s, c in zip(snake, camel))
print(f"{changed} of {len(snake)} field names change spelling")
# TODO: un-comment to pin it
# assert changed == 7

<details><summary>✅ Solution 13.1 — peek only after trying</summary>

```python
# my prediction: 7 of the 12 names change in camelCase
changed = sum(s != c for s, c in zip(snake, camel))
print(f"{changed} of {len(snake)} field names change spelling")
assert changed == 7
```

Exactly the seven snake_case names with an underscore change; single-word names
(`provider`, `consumer`, `params`, `price`, `salt`) spell identically in both languages.

</details>

## 14. `CANONICAL_SIGNED_OFFER` — the envelope Bell actually sends

On the wire, the offer travels inside an envelope carrying two more things: Bell's
**signature** — 65 bytes that prove Bell, and only Bell, authored these twelve fields
(how a key produces one is 06's job) — and the human-readable `terms_doc` whose
fingerprint you met in §9.

The fixture's signature is `0xabab…`: a syntactically valid **placeholder**, because
real signing needs Bell's private key, and keys live *only* in `chainmcp` (hard rule 2 —
this package couldn't sign even if it wanted to). The cardboard fakes in 05 don't verify
signatures at all; the real contract in 07 rejects this placeholder instantly.

In [ ]:
so = fx.CANONICAL_SIGNED_OFFER
print("signature:", so.signature[:20] + "…", f"({(len(so.signature) - 2) // 2} bytes)")
print("terms_doc:", so.terms_doc)
assert so.signature == "0x" + "ab" * 65
assert so.offer is fx.CANONICAL_OFFER      # the SAME object, not a copy — one source of truth
print("✓ envelope = offer + signature + human-readable terms")

**Break it on purpose.** 65 is not a round number by accident — a signature is three
parts (they get their names, `r`, `s`, `v`, in 06). 64 bytes is a *different, wrong*
thing, and the border knows it:

In [ ]:
from a2a_interfaces.models import SignedOffer

try:
    SignedOffer(offer=fx.CANONICAL_OFFER, signature="0x" + "ab" * 64, terms_doc={})
except ValidationError as e:
    print("✗ 64-byte 'signature' rejected:", e.errors()[0]["type"])
print("✓ exactly 65 bytes = r (32) + s (32) + v (1)")

## 15. `CANONICAL_ENTITLEMENT_VIEW` — the ticket, read back from the chain

The moment `fulfill()` settles the sale, the *offer* dies (salt punched, single-use)
and a *ticket* is born. `CANONICAL_ENTITLEMENT_VIEW` is ticket #7 as the controller
reads it back from the chain. It is the Offer's twin — but **not** its copy. Diff the
field names and stare at what changed:

In [ ]:
ev = fx.CANONICAL_ENTITLEMENT_VIEW
offer_fields = set(fx.CANONICAL_OFFER.model_dump())
view_fields = set(ev.model_dump())

print("disappeared:", sorted(offer_fields - view_fields))
print("appeared   :", sorted(view_fields - offer_fields))
print("kept       :", sorted(offer_fields & view_fields))
assert offer_fields - view_fields == {"provider", "consumer", "payment_token",
                                      "price", "valid_until", "salt"}
assert view_fields - offer_fields == {"id", "issuer", "revoked"}

Every change has a reason:

- `price`, `payment_token`, `valid_until`, `salt` — **payment concerns, spent at the
  moment of sale.** A cinema ticket doesn't say what you paid for it.
- `consumer` — gone because **ownership is now live data**: the ERC-721 `ownerOf(7)`
  answers "whose ticket?" at any moment, and the answer can change (tickets are
  transferable; a static field couldn't be).
- `provider` → **`issuer`** — same address (Bell), new role: no longer "the one
  offering" but "the one who issued it — and the only one allowed to `revoke` it".
- `id` = 7 — **didn't exist until mint**: it *is* the ticket number.
- `revoked` — a **live switch** only the issuer can flip; the controller's predicate
  checks it on every decision (08), and 12's finale flips it mid-session.

In [ ]:
print(f"id     : {ev.id}    issuer: {ev.issuer[:10]}…  (is BELL? {ev.issuer == fx.BELL})")
print(f"revoked: {ev.revoked}    window: {ev.start_time} → {ev.end_time}")
assert ev.issuer == fx.BELL and ev.id == fx.TICKET_ID and ev.revoked is False

**✏️ Your turn 15.1 — flip the live switch, on a copy**

Fixtures are frozen (§4), so revoke ticket #7 on a **copy**:
`ev.model_copy(update={"revoked": True})`. Success: the copy reads `revoked=True`
while the canon stays `False`.

In [ ]:
flipped = ev                          # TODO: replace with ev.model_copy(update={"revoked": True})
print(f"canon revoked: {ev.revoked}   copy revoked: {flipped.revoked}")
# TODO: un-comment once flipped is a real, revoked copy
# assert flipped.revoked is True and ev.revoked is False

<details><summary>✅ Solution 15.1 — peek only after trying</summary>

```python
flipped = ev.model_copy(update={"revoked": True})
print(f"canon revoked: {ev.revoked}   copy revoked: {flipped.revoked}")
assert flipped.revoked is True and ev.revoked is False
```

`model_copy(update=...)` builds a changed twin without touching the frozen original —
how tests simulate the revocation flip (12's finale does it for real, on chain).

</details>

Two kept fields changed their *type*: on the Offer, `resource_id` and `terms_hash` were
hex **strings** (`"0x…"`, JSON-friendly); on the view they are raw **bytes** — closer to
what the chain actually returns. Same 32 bytes, two skins (the `ser_json_bytes="hex"`
trick from 02 keeps JSON round-trips working). And `params` changed the most: opaque
hex blob → decoded, typed `BandwidthParams`.

In [ ]:
print("offer.resource_id:", type(fx.CANONICAL_OFFER.resource_id).__name__,
      "→", fx.CANONICAL_OFFER.resource_id[:12] + "…")
print("view.resource_id :", type(ev.resource_id).__name__, "         →", ev.resource_id[:6], "…")
assert ev.resource_id.hex() == fx.RESOURCE_ID[2:]     # identical bytes under both skins
assert ev.terms_hash.hex() == fx.TERMS_HASH[2:]

print("offer.params:", fx.CANONICAL_OFFER.params[:26] + "…   ← opaque hex blob")
print("view.params :", ev.params, "  ← decoded, typed")
assert ev.params.capacity_bps == fx.CAPACITY_50_MBPS and ev.params.qos_class == fx.QOS_CLASS

## 16. `BANDWIDTH_PARAMS_ABI` — dissect the blob by hand

How did `50_000_000` and `1` hide inside that hex blob? **ABI** (application binary
interface) encoding is how Ethereum packs typed values into bytes, and for simple values
the scheme is almost comically rigid: **every value gets exactly 32 bytes,
right-aligned** — the number sits at the far right, zeros pad the left. Our blob encodes
`(uint64 capacityBps, uint8 qosClass)`: two 32-byte words, 64 bytes, 128 hex characters.
No library — slice it yourself:

In [ ]:
blob = fx.BANDWIDTH_PARAMS_ABI
body = blob[2:]                                    # drop the 0x skin
words = [body[i : i + 64] for i in range(0, len(body), 64)]
print(f"blob: {len(body)} hex chars = {len(body) // 2} bytes = {len(words)} words × 32 bytes\n")
for i, w in enumerate(words):
    print(f"word {i}: {w}")
assert len(words) == 2

Now read each word as a base-16 integer — `int(word, 16)` — and the canonical numbers
fall out of the zeros:

In [ ]:
capacity = int(words[0], 16)
qos = int(words[1], 16)
print(f"word 0 → {capacity:>10,}   ← CAPACITY_50_MBPS (the tail …{words[0][-8:]} = 0x{capacity:x})")
print(f"word 1 → {qos:>10,}   ← QOS_CLASS")
assert capacity == fx.CAPACITY_50_MBPS and qos == fx.QOS_CLASS
print("✓ you just ABI-decoded by hand — no library, no magic")

And the fixture *builds* the blob the same way in reverse — two zero-padded f-string
words glued together. Read the real source line, then rebuild it yourself:

In [ ]:
import inspect

src = inspect.getsource(fx)
line = next(l for l in src.splitlines() if l.startswith("BANDWIDTH_PARAMS_ABI"))
print("fixtures.py:", line)

rebuilt = "0x" + f"{fx.CAPACITY_50_MBPS:064x}" + f"{fx.QOS_CLASS:064x}"
assert rebuilt == fx.BANDWIDTH_PARAMS_ABI
print("✓ rebuilt byte-for-byte — encoding is just formatting; decoding is just parsing")

**✏️ Your turn 16.1 — encode a different sale, decode your own blob**

Build the params blob for **200 Mbps at qos class 2** — two zero-padded 32-byte words,
exactly like the fixture line above — and let the decode loop read it back. Success:
`200,000,000 bps, qos 2`.

In [ ]:
my_blob = fx.BANDWIDTH_PARAMS_ABI     # TODO: build your own: "0x" + f"{200_000_000:064x}" + f"{2:064x}"
my_words = [my_blob[2:][i : i + 64] for i in range(0, len(my_blob[2:]), 64)]
print("decoded:", f"{int(my_words[0], 16):,} bps, qos", int(my_words[1], 16))
# TODO: un-comment once your blob round-trips
# assert (int(my_words[0], 16), int(my_words[1], 16)) == (200_000_000, 2)

<details><summary>✅ Solution 16.1 — peek only after trying</summary>

```python
my_blob = "0x" + f"{200_000_000:064x}" + f"{2:064x}"
my_words = [my_blob[2:][i : i + 64] for i in range(0, len(my_blob[2:]), 64)]
print("decoded:", f"{int(my_words[0], 16):,} bps, qos", int(my_words[1], 16))
assert (int(my_words[0], 16), int(my_words[1], 16)) == (200_000_000, 2)
```

Encoding is formatting and decoding is parsing: the same two format specs produce a
valid blob for *any* `(capacity, qos)` pair — why the contract never needs to
understand the payload it stores.

</details>

## 17. The second service — telemetry, ticket #8

The thesis claims a **service-agnostic** pattern — and one service can't prove
"agnostic". So the canon carries two services from day one. The story's chapter 7
introduces **Tess**, the telemetry provider, and lands the punchline: *the same machine
sells a very different service by swapping one translator.* (The fixtures keep the cast
small — Bell issues both tickets — but the product is Tess's.)

The second sale: the right to have a router push its **live interface counters** to
your collector. In the need below, `sensor_paths` holds a **gNMI/YANG path** — a
filesystem-like address into the router's live state ("the statistics of interface
ethernet-1/1"); 09 teaches that path language properly. `collector_endpoint` is where
the samples land; `sample_interval_s` is how often.

In [ ]:
print("The telemetry need (kind =", fx.TELEMETRY_NEED.kind + "):")
fx.TELEMETRY_NEED.model_dump()

Its minted twin is ticket **#8** with `service_type = 1` and `TelemetryParams` — same
window, same issuer, completely different payload. Note the params contain a *list of
strings*: **dynamic** ABI types, which the rigid 32-byte trick of §16 cannot encode
alone (dynamic values need offset words) — one more reason the real codec lives in a
library inside `chainmcp` (07), not in hand-rolled f-strings. And check the
one-source-of-truth discipline again: the view's params are read *off* `TELEMETRY_NEED`.

In [ ]:
tv = fx.TELEMETRY_ENTITLEMENT_VIEW
print(f"id: {tv.id}   service_type: {tv.service_type}   params: {type(tv.params).__name__}")
print("sensor_paths:", tv.params.sensor_paths)
assert tv.id == fx.TELEMETRY_TICKET_ID == 8 and tv.service_type == 1
assert tv.params.sensor_paths == fx.TELEMETRY_NEED.sensor_paths        # derived, not re-typed
assert tv.params.collector_endpoint == fx.TELEMETRY_NEED.collector_endpoint
print("✓ params derive FROM the need fixture — one source of truth, again")

Finally, what the opaque names of §10 *resolve to* — the two answer shapes the
controller hands to `netctl`. Bandwidth resolves to a **path** (a device plus ingress
and egress interfaces); telemetry resolves to a **node** (just a device). Both name
`srl1`, a real SR Linux router in the lab topology (09). Per ADR-005, this mapping lives
in exactly one file: the controller's `resource_map.yaml`.

In [ ]:
print("RESOLVED_PATH:", fx.RESOLVED_PATH.model_dump(), "  ← where ticket #7 lands")
print("RESOLVED_NODE:", fx.RESOLVED_NODE.model_dump(), "  ← where ticket #8 lands")
assert fx.RESOLVED_PATH.device == fx.RESOLVED_NODE.device == "srl1"

## 18. `DECISION_ACCEPT` — the one canonical judgment

Everything you met today is deterministic data — arithmetic any auditor can re-check.
Exactly **two** moments in the whole system involve LLM judgment (hard rule 1: the
controller is *never* an LLM): the provider deciding whether to quote, and the consumer
deciding whether to accept. `DECISION_ACCEPT` is the canonical accept — the tiny
structured shape the consumer's LLM must return: a boolean and its reason, nothing
else. Notebook 05's skeleton *scripts* this answer; notebook 10 wires a real model into
the slot.

In [ ]:
print(fx.DECISION_ACCEPT.model_dump())
assert fx.DECISION_ACCEPT.accept is True
print("✓ the ONLY judgment shape in the canon — everything else today was arithmetic")

**✏️ Your turn 18.1 — write the canonical rejection's twin**

The canon ships only the accept. Build its opposite with the **real** class —
`type(fx.DECISION_ACCEPT)` is `DecisionOutput` — giving it `accept=False` and a
one-line reason. Success: a dump with `accept: False` and your reason.

In [ ]:
DecisionOutput = type(fx.DECISION_ACCEPT)   # the real judgment class, straight off the canon
rejection = fx.DECISION_ACCEPT              # TODO: replace with DecisionOutput(accept=False, reason="...")
print(rejection.model_dump())
# TODO: un-comment once rejection really rejects
# assert rejection.accept is False and rejection.reason

<details><summary>✅ Solution 18.1 — peek only after trying</summary>

```python
DecisionOutput = type(fx.DECISION_ACCEPT)
rejection = DecisionOutput(accept=False, reason="price exceeds my 10 TOK budget")
print(rejection.model_dump())
assert rejection.accept is False and rejection.reason
```

The judgment slot is this tiny by design: an LLM may only ever answer these two
fields (hard rule 1) — everything else in today's tour stays deterministic.

</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| grep'd the repo for `0xf39F` / `ticket #7` | one canonical example is a hard convention; divergent values are a bug class | every later notebook quotes these same numbers |
| parsed `ADA` into 20 raw bytes, counted its casing | addresses = key-derived account names; EIP-55 checksum | 06 derives an address from a private key |
| rendered `1757944800` as 14:00 UTC | unix time; absolute windows on chain time (ADR-004) | 08's predicate re-checks chain time on every action |
| computed 10 TOK = 10·10¹⁸ base units | integer money, `DecimalString` on the wire | 06 moves real TOK; 07 prices the real offer |
| watched sha-256's avalanche on a one-char edit | hashes as commitments; `termsHash` | 06 meets keccak; 07 hashes the real `terms_doc` |
| diffed `Offer` against `EntitlementView` | sale data dies at settlement; ticket data lives | 05 mints the fake #7; 07 mints the real one |
| ABI-decoded the params blob by hand | fixed-width 32-byte words, right-aligned | 07 lets a real codec do it (and handle dynamic types) |
| toured ticket #8, `ResolvedPath` vs `ResolvedNode` | service-agnostic needs two services (ADR-005 resolution) | 09 provisions both kinds against a router |

## Experiments to try

Predict the outcome *before* you run each one:

- **Bigger pipe.** If the capacity were 1 Gbps, what would §16's word 0 look like?
  Compute `f"{1_000_000_000:064x}"` and check where the nonzero tail starts compared to
  50 Mbps.
- **The validation loophole.** Run
  `fx.CANONICAL_OFFER.model_copy(update={"price": "10.5"})`. Will it raise like §6's
  break did? (02 warned you about this one.) Then catch the lie with
  `Offer.model_validate(bad.model_dump())`.
- **Your own clock.** Render `datetime.fromtimestamp(fx.WINDOW.start)` *without*
  `tz=timezone.utc`. Why do the clock digits differ from 14:00 while the instant in
  time is exactly the same?
- **Deliberate corruption.** Flip one hex character somewhere in the middle of
  `fx.BANDWIDTH_PARAMS_ABI` (make a new string — the fixture is frozen) and re-run the
  §16 decode. Which value gets corrupted — capacity or qos? Predict from the
  character's position before you look.

## Check yourself

1. Why does the repo forbid a doc from writing "Ada pays 12 TOK" in an example — what
   bug class does the one-source rule kill, and where is the one source?
2. The offer carries both `valid_until` (14:20) and `end_time` (16:00). What does each
   one expire, and who enforces each?
3. Six fields vanish between `Offer` and `EntitlementView`. Name three and explain why
   each has no business existing on the ticket.
4. `50_000_000` occupies a full 32-byte word in the params blob. Why does a `uint64`
   get 32 bytes, and what does "right-aligned" mean for the hex string?
5. Whose name is `resource_id` — who writes it, who resolves it, and who must never
   interpret it?

**Where this goes next:** `05_the_walking_skeleton.ipynb` 🎬 — the whole lifecycle runs
on cardboard fakes using exactly these numbers: the fake chain punches *this* salt,
mints *this* ticket #7, and a naive predicate judges *this* window.

**Deeper dive:**
- `e2e/notebooks/skeleton_explore.ipynb` — the working-engineer tour that drives these fixtures through a full lifecycle
- `docs/03a-interfaces-walkthrough.md` — the narrative walkthrough of the shapes you toured
- `docs/03-interfaces.md` §1.4 — the canonical values inside the formal schema
- `docs/00-the-story.md` (Epilogue) — the twelve-line story these numbers act out
- `interfaces/tests/test_fixtures.py` — the tests that pin the canon